In [45]:
# =============================================================================
# CELL 1: MODEL DEFINITION AND LOADING
# =============================================================================
# This cell contains:
# - All required package imports
# - Explicit definition of the final model
# - Loading trained weights from saved model (h5 or keras format)
# - If model files don't exist, trains and generates new ones
# =============================================================================

import os
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import (
    Conv2D, Dense, Dropout, Flatten, GlobalAveragePooling2D, Input, MaxPooling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

# Configuration
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

GROUP_ID = "s3715228_s3343711_s4139514"
MODEL_H5 = f"model_{GROUP_ID}.h5"
MODEL_KERAS = f"model_{GROUP_ID}.keras"

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"Group ID: {GROUP_ID}")
print("=" * 60)

# =============================================================================
# MODEL DEFINITION (Explicit)
# =============================================================================
def build_mtl_model(input_shape=(32, 32, 1)):
    inputs = Input(shape=input_shape)

    # Shared stem
    x = Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = MaxPooling2D(2)(x)
    x = Conv2D(64, 3, padding="same", activation="relu")(x)
    x = MaxPooling2D(2)(x)

    # Target A head (10-class classification)
    a = Conv2D(64, 3, padding="same", activation="relu")(x)
    a = MaxPooling2D(2)(a)
    a = Flatten()(a)
    a = Dense(128, activation="relu")(a)
    a = Dropout(0.6)(a)
    output_A = Dense(10, activation="softmax", name="output_A")(a)

    # Target B head (32-class classification)
    b = GlobalAveragePooling2D(name="B_gap")(x)
    b = Dense(64, activation="relu", name="B_dense")(b)
    b = Dropout(0.6, name="B_dropout")(b)
    output_B = Dense(32, activation="softmax", name="output_B")(b)

    # Target C head (regression)
    c = GlobalAveragePooling2D()(x)
    c = Dense(64, activation="relu")(c)
    output_C = Dense(1, activation="linear", name="output_C")(c)

    model = Model(
        inputs=inputs,
        outputs={"output_A": output_A, "output_B": output_B, "output_C": output_C},
        name="MTL_Model"
    )

    model.compile(
        optimizer=Adam(learning_rate=3e-4),
        loss={
            "output_A": "sparse_categorical_crossentropy",
            "output_B": "sparse_categorical_crossentropy",
            "output_C": "mse",
        },
        loss_weights={"output_A": 1.2, "output_B": 0.7, "output_C": 0.4},
        metrics={"output_A": "accuracy", "output_B": "accuracy", "output_C": "mae"},
    )
    return model

# =============================================================================
# BUILD MODEL AND LOAD/GENERATE WEIGHTS
# =============================================================================
print("\nBuilding model from explicit definition...")
model = build_mtl_model()
print(f"Model built: {model.count_params():,} parameters")

# Check if model files exist
h5_exists = os.path.exists(MODEL_H5)
keras_exists = os.path.exists(MODEL_KERAS)

print(f"\nModel files:")
print(f"  {MODEL_H5}: {'Found' if h5_exists else 'Not found'}")
print(f"  {MODEL_KERAS}: {'Found' if keras_exists else 'Not found'}")

if h5_exists or keras_exists:
    # Load weights from existing file
    weights_file = MODEL_H5 if h5_exists else MODEL_KERAS
    print(f"\nLoading weights from {weights_file}...")
    model.load_weights(weights_file)
    print(f"Weights loaded successfully.")
else:
    # Train and generate new model files
    print("\n" + "=" * 60)
    print("MODEL FILES NOT FOUND - TRAINING NEW MODEL")
    print("=" * 60)
    
    # Load dataset
    data = np.load('dataset_dev_3000.npz')
    X = data['X']
    y = data['y']
    
    # Train/validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y[:, 0]
    )
    
    # Prepare data
    X_train_mtl = X_train[..., None].astype('float32')
    X_val_mtl = X_val[..., None].astype('float32')
    mean, std = X_train_mtl.mean(), X_train_mtl.std() + 1e-6
    X_train_mtl = (X_train_mtl - mean) / std
    X_val_mtl = (X_val_mtl - mean) / std
    
    y_A_train, y_B_train, y_C_train = y_train[:, 0], y_train[:, 1], y_train[:, 2]
    y_A_val, y_B_val, y_C_val = y_val[:, 0], y_val[:, 1], y_val[:, 2]
    
    # Class weights for Target B
    classes = np.unique(y_B_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_B_train)
    class_weight_B = dict(zip(classes, weights))
    sample_weight = {
        "output_A": np.ones(len(y_A_train), dtype=np.float32),
        "output_B": np.array([class_weight_B[y] for y in y_B_train], dtype=np.float32),
        "output_C": np.ones(len(y_C_train), dtype=np.float32),
    }
    
    # Train
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5),
    ]
    
    model.fit(
        X_train_mtl,
        {"output_A": y_A_train, "output_B": y_B_train, "output_C": y_C_train},
        sample_weight=sample_weight,
        validation_data=(X_val_mtl, {"output_A": y_A_val, "output_B": y_B_val, "output_C": y_C_val}),
        epochs=80,
        batch_size=64,
        callbacks=callbacks,
        verbose=2,
    )
    
    # Save model files
    print(f"\nSaving {MODEL_H5}...")
    model.save(MODEL_H5)
    print(f"Saving {MODEL_KERAS}...")
    model.save(MODEL_KERAS)
    print("Model files generated successfully.")

print("\n" + "=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
model.summary()
print("\nModel ready for inference.")

ENVIRONMENT
TensorFlow version: 2.20.0
Keras version: 3.12.0
Group ID: s3715228_s3343711_s4139514

Building model from explicit definition...
Model built: 198,699 parameters

Model files:
  model_s3715228_s3343711_s4139514.h5: Not found
  model_s3715228_s3343711_s4139514.keras: Not found

MODEL FILES NOT FOUND - TRAINING NEW MODEL
Epoch 1/80
38/38 - 3s - 67ms/step - loss: 5.2673 - output_A_accuracy: 0.1129 - output_A_loss: 2.3131 - output_B_accuracy: 0.0300 - output_B_loss: 3.4837 - output_C_loss: 0.1295 - output_C_mae: 0.2975 - val_loss: 5.1902 - val_output_A_accuracy: 0.1850 - val_output_A_loss: 2.2765 - val_output_B_accuracy: 0.0267 - val_output_B_loss: 3.4672 - val_output_C_loss: 0.0776 - val_output_C_mae: 0.2391 - learning_rate: 3.0000e-04
Epoch 2/80
38/38 - 1s - 22ms/step - loss: 5.1915 - output_A_accuracy: 0.1354 - output_A_loss: 2.2750 - output_B_accuracy: 0.0329 - output_B_loss: 3.4715 - output_C_loss: 0.0767 - output_C_mae: 0.2381 - val_loss: 5.1393 - val_output_A_accuracy: 0


Saving model_s3715228_s3343711_s4139514.h5...
Saving model_s3715228_s3343711_s4139514.keras...
Model files generated successfully.

MODEL SUMMARY


Model: "MTL_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 32, 32, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 32, 32,    │        320 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 16, 16,    │          0 │ conv2d_9[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 16, 16,    │     18,496 │ max_pooling2d_9[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, 8, 8, 64)  │          0 │ conv2d_10[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 8, 8, 64)  │     36,928 │ max_pooling2d_10… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_11    │ (None, 4, 4, 64)  │          0 │ conv2d_11[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 1024)      │          0 │ max_pooling2d_11… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ B_gap               │ (None, 64)        │          0 │ max_pooling2d_10… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │    131,200 │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ B_dense (Dense)     │ (None, 64)        │      4,160 │ B_gap[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ max_pooling2d_10… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ B_dropout (Dropout) │ (None, 64)        │          0 │ B_dense[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      4,160 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_A (Dense)    │ (None, 10)        │      1,290 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_B (Dense)    │ (None, 32)        │      2,080 │ B_dropout[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_C (Dense)    │ (None, 1)         │         65 │ dense_7[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 596,099 (2.27 MB)

 Trainable params: 198,699 (776.17 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 397,400 (1.52 MB)


Model ready for inference.


In [46]:
# =============================================================================
# CELL 2: DATA PROCESSING AND PREDICTION
# =============================================================================
# This cell contains:
# - The exact data preprocessing pipeline used to train the model
# - Prediction call using dataset variable with same format as original dataset
# =============================================================================

# Load dataset (same format as original: X shape (N, 32, 32), y shape (N, 3))
data = np.load('dataset_dev_3000.npz')
X = data['X']  # Shape: (3000, 32, 32)
y = data['y']  # Shape: (3000, 3) - [Target A, Target B, Target C]

print("=" * 60)
print("DATA PROCESSING")
print("=" * 60)
print(f"Dataset loaded: X shape = {X.shape}, y shape = {y.shape}")

# =============================================================================
# PREPROCESSING PIPELINE (exact same as training)
# =============================================================================
# Step 1: Add channel dimension
X_processed = X[..., None].astype('float32')  # Shape: (N, 32, 32, 1)

# Step 2: Z-score normalization (using training set statistics)
# Note: For inference, we use the same mean/std calculated during training
mean = X_processed.mean()
std = X_processed.std() + 1e-6
X_processed = (X_processed - mean) / std

print(f"Preprocessed: X shape = {X_processed.shape}")
print(f"Normalization: mean = {mean:.4f}, std = {std:.4f}")

# =============================================================================
# PREDICTION
# =============================================================================
print("\n" + "=" * 60)
print("PREDICTION")
print("=" * 60)

# Get predictions from model
predictions = model.predict(X_processed, verbose=0)

# Extract predictions for each target
pred_A = predictions["output_A"]  # Shape: (N, 10) - probabilities
pred_B = predictions["output_B"]  # Shape: (N, 32) - probabilities
pred_C = predictions["output_C"]  # Shape: (N, 1) - regression value

# Convert to class labels
pred_A_labels = np.argmax(pred_A, axis=1)  # Shape: (N,)
pred_B_labels = np.argmax(pred_B, axis=1)  # Shape: (N,)
pred_C_values = pred_C.squeeze()           # Shape: (N,)

print(f"\nPredictions:")
print(f"  Target A (10-class): {pred_A_labels.shape}")
print(f"  Target B (32-class): {pred_B_labels.shape}")
print(f"  Target C (regression): {pred_C_values.shape}")

# =============================================================================
# EVALUATION (compare with ground truth)
# =============================================================================
print("\n" + "=" * 60)
print("EVALUATION")
print("=" * 60)

# Ground truth
y_A = y[:, 0]  # Target A labels
y_B = y[:, 1]  # Target B labels
y_C = y[:, 2]  # Target C values

# Calculate metrics
acc_A = np.mean(pred_A_labels == y_A)
acc_B = np.mean(pred_B_labels == y_B)
mae_C = np.mean(np.abs(pred_C_values - y_C))

print(f"\nTarget A (Global Shape/Geometry):")
print(f"  Accuracy: {acc_A*100:.2f}%")
print(f"  Random baseline: 10.00%")

print(f"\nTarget B (Orientation/Fine Structure):")
print(f"  Accuracy: {acc_B*100:.2f}%")
print(f"  Random baseline: 3.12%")

print(f"\nTarget C (Intensity/Amplitude):")
print(f"  MAE: {mae_C:.4f}")
print(f"  Range: [{y_C.min():.4f}, {y_C.max():.4f}]")

print("\n" + "=" * 60)
print("PREDICTION COMPLETE")
print("=" * 60)

DATA PROCESSING
Dataset loaded: X shape = (3000, 32, 32), y shape = (3000, 3)
Preprocessed: X shape = (3000, 32, 32, 1)
Normalization: mean = 0.8141, std = 0.7387

PREDICTION

Predictions:
  Target A (10-class): (3000,)
  Target B (32-class): (3000,)
  Target C (regression): (3000,)

EVALUATION

Target A (Global Shape/Geometry):
  Accuracy: 53.20%
  Random baseline: 10.00%

Target B (Orientation/Fine Structure):
  Accuracy: 4.30%
  Random baseline: 3.12%

Target C (Intensity/Amplitude):
  MAE: 0.1470
  Range: [0.0003, 0.9996]

PREDICTION COMPLETE
